# 1) Transform Data to have **normalized values** & **sov-based countries**
- normalized values: floats in \[0, 1\]
- countries need to be named with their SOV code


In [ ]:
from typing import Dict

import os
import pandas as pd

from pain2map import Util

In [ ]:
BASE_PATH = os.path.join("..", "data")
DATASET_BASE_PATH = os.path.join(BASE_PATH, "2")
DATASET_PATH = os.path.join(DATASET_BASE_PATH, "GDP.csv")
NORM_DATASET_PATH = os.path.join(DATASET_BASE_PATH, "GDP_normalized.csv")
SOVA3_DATASET_PATH = os.path.join(DATASET_BASE_PATH, "GDP_sova3.csv")

WORLD_PATH = os.path.join(BASE_PATH, "countries_map.zip")

In [ ]:
STORE_CSVS = True

In [ ]:
dataset = pd.read_csv(DATASET_PATH)
dataset.head()

## 1.1) Normalize values

this step heavily depends on the dataset

In [ ]:
# sum all numeric columns for each country and divide by the max sum to get a value between 0 and 1
value_columns = [col for col in dataset.columns.values.tolist() if col.isnumeric()]
result_data = []
max_sum = 0.0
for index, row in dataset.iterrows():
    country = row["Country"]
    cur_sum = 0.0
    for col in value_columns:
        value = row[col]
        if pd.isna(value):
            continue
        cur_sum += float(value)
    if max_sum < cur_sum:
        max_sum = cur_sum
    result_data.append({"Country": country, "value": cur_sum})
for item in result_data:
    item["value"] /= max_sum
norm_dataset = pd.DataFrame(result_data)
if STORE_CSVS:
    norm_dataset.to_csv(NORM_DATASET_PATH, index=False)

## 1.2) Transform to SOV-based Countries

In [ ]:

if STORE_CSVS:
    norm_dataset = pd.read_csv(NORM_DATASET_PATH)
sov_dataset, sov_subset = Util.compute_sova3_subset(norm_dataset, WORLD_PATH, "Country", "value")
if STORE_CSVS:
    sov_dataset.to_csv(SOVA3_DATASET_PATH, index=False)
sov_dataset.head()

# 2) Generate Map from Dataset

In [ ]:
if STORE_CSVS:
    sov_dataset = pd.read_csv(SOVA3_DATASET_PATH)
Util.generate_map(
    data=sov_dataset,
    world_path=WORLD_PATH,
    output_path=os.path.join(BASE_PATH, "2", "GDP_map.png"),
    args_value_col="value",
    args_code_col="sov_a3",
    width=1024,
    dpi=100,
    cmap="viridis",
    projection="PlateCarree"
)
print("Done")